In [ ]:
#%pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 11.4 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 11.2 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 11.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 545.8/545.8 kB 8.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 11.5 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 11.3 MB/s  0:00:00
  Attempting uninstall: yarl
    Found existing installation: yarl 1.22.0
    Uninstalling yarl-1.22.0:
      Successfully uninstalled yarl-1.22.0
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0━━━━━━━━━━━━━━━━━━━━━━  6/37 [urllib3]
  Attempting uninstall: typing-extensions━━━━━━━━━━━━━━━━━━━━━  6/37 [urllib3]
    Found existing installation: typing_extensions 4.15.0━━━━━  6/37 [urllib3]

In [3]:
import os, re, json, textwrap
from pathlib import Path
import numpy as np
import requests
import chromadb

try:
    from tqdm import tqdm          # progress bar for the one-time embed (optional)
except ImportError:
    tqdm = None

# --- config (same values as the core notebook, so the store matches whether we
#     reuse the one it built or build an identical one ourselves) ---
OLLAMA_URL  = "http://localhost:11434"
CHAT_MODEL  = "qwen3.5:9b"
EMBED_MODEL = "embeddinggemma:latest"
TOPK        = 5

WORDS_PER_CHUNK  = 300       # only used on the self-build path below
OVERLAP_WORDS    = 60
EMBED_BATCH_SIZE = 64
CORPUS_DIR        = Path("corpus_jupyter")
CHROMA_PATH       = "./chroma_db"
CHROMA_COLLECTION = "rag_demo"
DOWNLOAD_FROM_WEB = True

SESSION = requests.Session()
SESSION.trust_env = False        # bypass VPN/proxy for the localhost Ollama calls

# --- open the SAME collection the core notebook uses (the foundation we reuse) ---
collection = chromadb.PersistentClient(path=CHROMA_PATH).get_or_create_collection(
    CHROMA_COLLECTION, metadata={"hnsw:space": "cosine"})

already_built = collection.count() > 0
print(f"Chroma collection '{CHROMA_COLLECTION}': {collection.count()} vectors.")
print("Already populated — we'll reuse it." if already_built
      else "Empty — the next cell will build it from scratch.")

Chroma collection 'rag_demo': 0 vectors.
Empty — the next cell will build it from scratch.


In [6]:
if already_built:
    print(f"Reusing '{CHROMA_COLLECTION}' — {collection.count()} vectors already on disk. "
          "Nothing to build.")
else:
    print("Empty collection — building it now (same pipeline as the core notebook)...\n")

    GUTENBERG_BOOKS = {
        "Moby-Dick": "https://www.gutenberg.org/files/2701/2701-0.txt",
        "Pride and Prejudice": "https://www.gutenberg.org/files/1342/1342-0.txt",
        "Frankenstein": "https://www.gutenberg.org/files/84/84-0.txt",
        "Alice in Wonderland": "https://www.gutenberg.org/cache/epub/11/pg11.txt",
        "Dracula": "https://www.gutenberg.org/files/345/345-0.txt",
        "A Tale of Two Cities": "https://www.gutenberg.org/files/98/98-0.txt",
        "The Great Gatsby": "https://www.gutenberg.org/cache/epub/64317/pg64317.txt",
        "Adventures of Sherlock Holmes": "https://www.gutenberg.org/files/1661/1661-0.txt",
        "War and Peace": "https://www.gutenberg.org/files/2600/2600-0.txt",
        "Jane Eyre": "https://www.gutenberg.org/files/1260/1260-0.txt",
        "The Picture of Dorian Gray": "https://www.gutenberg.org/files/174/174-0.txt",
        "Crime and Punishment": "https://www.gutenberg.org/files/2554/2554-0.txt",
        "Wuthering Heights": "https://www.gutenberg.org/files/768/768-0.txt",
    }

    # 1. Download each book; strip Project Gutenberg's license header/footer.
    CORPUS_DIR.mkdir(parents=True, exist_ok=True)
    START_MARK = re.compile(r"\*\*\* START OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)
    END_MARK   = re.compile(r"\*\*\* END OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)

    docs = []
    for title, url in GUTENBERG_BOOKS.items():
        path = CORPUS_DIR / (re.sub(r"[^A-Za-z0-9]+", "_", title).strip("_") + ".txt")
        if not path.exists():
            if not DOWNLOAD_FROM_WEB:
                continue
            print(f"  downloading: {title}")
            resp = SESSION.get(url, timeout=180)
            resp.raise_for_status()
            raw = resp.text
            s, e = START_MARK.search(raw), END_MARK.search(raw)
            if s and e and e.start() > s.end():
                raw = raw[s.end():e.start()]
            path.write_text(raw.strip(), encoding="utf-8")
        docs.append({"title": title, "url": url,
                     "text": path.read_text(encoding="utf-8", errors="ignore")})

    # 2. Chunk each book into overlapping word windows (id like "frankenstein#7").
    chunks = []
    for d in docs:
        doc_id = re.sub(r"[^a-z0-9]+", "-", d["title"].lower()).strip("-")
        words = re.sub(r"\s+", " ", d["text"]).strip().split()
        step = max(1, WORDS_PER_CHUNK - OVERLAP_WORDS)
        i = 0
        for start in range(0, len(words), step):
            window = words[start:start + WORDS_PER_CHUNK]
            if len(window) < max(60, WORDS_PER_CHUNK // 4):
                break
            chunks.append({"id": f"{doc_id}#{i}", "doc_id": doc_id, "chunk_index": i,
                           "title": d["title"], "source": d["url"], "text": " ".join(window)})
            i += 1
            if start + WORDS_PER_CHUNK >= len(words):
                break

    # 3. Embed every chunk and store it in Chroma (the slow, one-time step).
    batches = range(0, len(chunks), EMBED_BATCH_SIZE)
    if tqdm:
        batches = tqdm(batches, desc=f"Embedding with {EMBED_MODEL}")
    for sidx in batches:
        batch = chunks[sidx:sidx + EMBED_BATCH_SIZE]
        texts = [c["text"] for c in batch]
        resp = SESSION.post(f"{OLLAMA_URL}/api/embed",
                            json={"model": EMBED_MODEL, "input": texts}, timeout=600)
        resp.raise_for_status()
        vectors = resp.json()["embeddings"]
        collection.add(ids=[c["id"] for c in batch], documents=texts,
                       metadatas=[{"doc_id": c["doc_id"], "chunk_index": c["chunk_index"],
                                   "title": c["title"], "source": c["source"]} for c in batch],
                       embeddings=vectors)

    already_built = True
    print(f"\nDone — Chroma now holds {collection.count()} vectors.")

Empty collection — building it now (same pipeline as the core notebook)...

  downloading: Moby-Dick


Embedding with embeddinggemma:latest: 100%|██████████| 133/133 [05:39<00:00,  2.55s/it]


Done — Chroma now holds 8510 vectors.


In [7]:
# --- the toolkit: the core notebook's steps, packaged as 5 small helpers ---
def ollama_chat(messages, temperature=0.2):
    """Call the local chat model and return its text reply."""
    r = SESSION.post(f"{OLLAMA_URL}/api/chat",
                     json={"model": CHAT_MODEL, "messages": messages,
                           "stream": False, "options": {"temperature": temperature}},
                     timeout=600)
    r.raise_for_status()
    return r.json()["message"]["content"]


def dense_search(query, k=TOPK, where=None):
    """Embed `query`, return up to k Chroma hits (optionally filtered by `where`)."""
    r = SESSION.post(f"{OLLAMA_URL}/api/embed",
                     json={"model": EMBED_MODEL, "input": [query]}, timeout=600)
    r.raise_for_status()
    q_vec = r.json()["embeddings"][0]
    res = collection.query(query_embeddings=[q_vec], n_results=k, where=where,
                           include=["documents", "metadatas", "distances"])
    hits = []
    for id_, doc, meta, dist in zip(res["ids"][0], res["documents"][0],
                                    res["metadatas"][0], res["distances"][0]):
        meta = meta or {}
        hits.append({"id": id_, "text": doc or "", "title": meta.get("title", ""),
                     "doc_id": meta.get("doc_id", ""), "chunk_index": meta.get("chunk_index", -1),
                     "distance": float(dist)})
    return hits


def build_context(hits):
    """Stitch hits into one context block, each tagged with [doc_id#chunk_index]."""
    return "".join(f"[{h['doc_id']}#{h['chunk_index']}] {h['title']}\n{h['text']}\n---\n"
                   for h in hits) or "NO_CONTEXT"


SYSTEM_PROMPT = (
    "You are a helpful assistant. Answer ONLY using the provided context. "
    "If the answer is not in the context, say: 'I don't know based on the provided context.' "
    "Cite sources like [doc_id#chunk_index] for each key claim."
)


def generate(question, hits, temperature=0.2):
    """Answer `question` grounded only in `hits`."""
    user = f"Question: {question}\n\nContext:\n{build_context(hits)}"
    return ollama_chat([{"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": user}], temperature)


def show_hits(hits, key="distance"):
    """Pretty-print a hit list; `key` picks which score to display."""
    for i, h in enumerate(hits, 1):
        score = f"  {key}={h[key]:.3f}" if key in h and h[key] is not None else ""
        print(f"  {i}. {h['title']} [{h['doc_id']}#{h['chunk_index']}]{score}")

In [8]:
dense_search("How do I install the package?", k=3)

[{'id': 'dracula#621',
  'text': 'he could go either as man, or wolf, or bat, or in some other way. He evidently fears discovery or interference, in the state of helplessness in which he must be--confined as he is between dawn and sunset in his wooden box. (_b_) _How is he to be taken?_--Here a process of exclusions may help us. By road, by rail, by water? 1. _By Road._--There are endless difficulties, especially in leaving the city. (_x_) There are people; and people are curious, and investigate. A hint, a surmise, a doubt as to what might be in the box, would destroy him. (_y_) There are, or there may be, customs and octroi officers to pass. (_z_) His pursuers might follow. This is his highest fear; and in order to prevent his being betrayed he has repelled, so far as he can, even his victim--me! 2. _By Rail._--There is no one in charge of the box. It would have to take its chance of being delayed; and delay would be fatal, with enemies on the track. True, he might escape at night; b

## How to apply metadata filtering 

In [9]:
query = "a mysterious creature comes to life"

print("Unfiltered (any book):")
show_hits(dense_search(query, k=3))

Unfiltered (any book):
  1. Frankenstein [frankenstein#92]  distance=0.599
  2. Adventures of Sherlock Holmes [adventures-of-sherlock-holmes#420]  distance=0.601
  3. Jane Eyre [jane-eyre#735]  distance=0.609


In [10]:
print("\nScoped to 'Frankenstein' only:")
show_hits(dense_search(query, k=3, where={"title": "Frankenstein"}))


Scoped to 'Frankenstein' only:
  1. Frankenstein [frankenstein#92]  distance=0.599
  2. Frankenstein [frankenstein#61]  distance=0.613
  3. Frankenstein [frankenstein#16]  distance=0.621


# Query transformations 

In [11]:
# --- Multi-query: paraphrase the question, retrieve each, union by best distance ---
question = "Elizabeth's changing feelings about Darcy's proposal"

prompt = (f"Generate 3 alternative search queries (different wording / aspects) for the "
          f"question. The search query should typically have the words used in novels, so it has a higher similarity. One per line, no numbering.\n\nQuestion: {question}")
raw = ollama_chat([{"role": "user", "content": prompt}], temperature=0.5)
variants = ([question] + [ln.strip("-* ").strip() for ln in raw.splitlines() if ln.strip()])[:4]

print("Search queries:")
for v in variants:
    print("  -", v)

best = {}                                    # id -> best (lowest-distance) hit across all variants
for v in variants:
    for h in dense_search(v, k=5):
        if h["id"] not in best or h["distance"] < best[h["id"]]["distance"]:
            best[h["id"]] = h

print("\nUnion of results:")
show_hits(sorted(best.values(), key=lambda h: h["distance"])[:TOPK])

Search queries:
  - Elizabeth's changing feelings about Darcy's proposal
  - Elizabeth Bennet reaction to Darcy's first proposal in Pride and Prejudice
  - How does Elizabeth's perspective shift after rejecting Darcy's offer
  - Elizabeth's evolving sentiments toward Darcy following his initial proposal

Union of results:
  1. Pride and Prejudice [pride-and-prejudice#271]  distance=0.458
  2. Pride and Prejudice [pride-and-prejudice#518]  distance=0.464
  3. Pride and Prejudice [pride-and-prejudice#457]  distance=0.475
  4. Pride and Prejudice [pride-and-prejudice#265]  distance=0.477
  5. Pride and Prejudice [pride-and-prejudice#500]  distance=0.483


### HyDE (Hypothetical Document Embeddings)

In [12]:
# --- HyDE: hallucinate an answer, embed THAT, and search with it ---
question = "How does the creature feel toward Victor Frankenstein?"

hypo = ollama_chat([{"role": "user",
                     "content": f"Write a short, factual paragraph answering: {question}"}],
                   temperature=0.3)
print("Hypothetical answer we'll search with:")
print(textwrap.fill(hypo, width=88), "\n")

# dense_search embeds whatever text we hand it — so we just search with the hypo paragraph.
show_hits(dense_search(hypo))

Hypothetical answer we'll search with:
The creature’s feelings toward Victor Frankenstein evolve from initial affection and
hope to profound hatred and vengeance. Initially viewing Victor as a creator and father
figure, the creature seeks guidance and companionship but is met with horror and
rejection upon his appearance. This abandonment fuels a deep sense of betrayal, leading
the creature to blame Victor for his isolation and suffering. Consequently, their
relationship transforms into one of bitter animosity, where the creature pursues
retribution against his creator. 

  1. Frankenstein [frankenstein#126]  distance=0.388
  2. Frankenstein [frankenstein#309]  distance=0.436
  3. Frankenstein [frankenstein#125]  distance=0.447
  4. Frankenstein [frankenstein#306]  distance=0.450
  5. Frankenstein [frankenstein#305]  distance=0.454


### Decompose

In [13]:
# --- Decompose (split into sub-questions) and step-back (ask broader) ---
question = "Compare how Darcy and Heathcliff express love."

decomp = ollama_chat([{"role": "user", "content":
    f"Break this into up to 3 simpler standalone sub-questions, one per line. "
    f"If already simple, return it unchanged.\n\nQuestion: {question}"}], temperature=0.3)
print("Decomposed into sub-questions:")
for ln in decomp.splitlines():
    if ln.strip():
        print("  -", ln.strip("-*0123456789. ").strip())

stepback = ollama_chat([{"role": "user", "content":
    f"Give ONE broader background question behind this one:\n{question}"}], temperature=0.3)
print("\nStep-back question:", stepback.strip().splitlines()[0])

Decomposed into sub-questions:
  - How does Darcy express his love?
  - How does Heathcliff express his love?
  - What are the similarities and differences between their expressions of love?

Step-back question: How do social class and historical context influence the portrayal of romantic relationships in 19th-century English literature?


### Hybrid Search

#culminated_q_top_40k_fintech = .... 

BM25 stands for Best Match 25. What this model does is it does a keyword-based search. 

In [18]:
try:
    from rank_bm25 import BM25Okapi
except ImportError:
    raise ImportError("pip install rank_bm25  (see the optional-deps cell at the top)")

# Pull every stored chunk back out of Chroma — no external chunks.json needed.
store = collection.get(include=["documents", "metadatas"])
bm25_ids   = store["ids"]
bm25_texts = store["documents"]
bm25_metas = store["metadatas"]

bm25 = BM25Okapi([t.lower().split() for t in bm25_texts])
print(f"BM25 index built over {len(bm25_texts)} chunks pulled from Chroma.")

BM25 index built over 8510 chunks pulled from Chroma.


In [19]:
bm25_texts[0]

'MOBY-DICK; or, THE WHALE. By Herman Melville CONTENTS ETYMOLOGY. EXTRACTS (Supplied by a Sub-Sub-Librarian). CHAPTER 1. Loomings. CHAPTER 2. The Carpet-Bag. CHAPTER 3. The Spouter-Inn. CHAPTER 4. The Counterpane. CHAPTER 5. Breakfast. CHAPTER 6. The Street. CHAPTER 7. The Chapel. CHAPTER 8. The Pulpit. CHAPTER 9. The Sermon. CHAPTER 10. A Bosom Friend. CHAPTER 11. Nightgown. CHAPTER 12. Biographical. CHAPTER 13. Wheelbarrow. CHAPTER 14. Nantucket. CHAPTER 15. Chowder. CHAPTER 16. The Ship. CHAPTER 17. The Ramadan. CHAPTER 18. His Mark. CHAPTER 19. The Prophet. CHAPTER 20. All Astir. CHAPTER 21. Going Aboard. CHAPTER 22. Merry Christmas. CHAPTER 23. The Lee Shore. CHAPTER 24. The Advocate. CHAPTER 25. Postscript. CHAPTER 26. Knights and Squires. CHAPTER 27. Knights and Squires. CHAPTER 28. Ahab. CHAPTER 29. Enter Ahab; to Him, Stubb. CHAPTER 30. The Pipe. CHAPTER 31. Queen Mab. CHAPTER 32. Cetology. CHAPTER 33. The Specksnyder. CHAPTER 34. The Cabin-Table. CHAPTER 35. The Mast-Head. CH

In [20]:
# "Pequod" is a rare proper noun -> keyword search shines; pure semantics can drift.
query = "What is the Pequod?"
pool = 10

# BM25 hits (keyword): score every chunk, take the top `pool`.
scores = bm25.get_scores(query.lower().split())
top = np.argsort(scores)[::-1][:pool]
bm25_hits = [{"id": bm25_ids[i], "text": bm25_texts[i],
              "title": bm25_metas[i].get("title", ""), "doc_id": bm25_metas[i].get("doc_id", ""),
              "chunk_index": bm25_metas[i].get("chunk_index", -1), "bm25": float(scores[i])}
             for i in top]


# Dense hits (semantic) via the toolkit.
dense_hits = dense_search(query, k=pool)

# Reciprocal Rank Fusion: each list contributes 1/(k + rank) to a chunk's score.
RRF_K = 60
fused, seen = {}, {}
for ranked in (dense_hits, bm25_hits):
    for rank, h in enumerate(ranked):
        fused[h["id"]] = fused.get(h["id"], 0.0) + 1.0 / (RRF_K + rank + 1)
        seen.setdefault(h["id"], h)
hybrid_hits = [{**seen[i], "rrf": s}
               for i, s in sorted(fused.items(), key=lambda x: -x[1])[:5]]

print("Dense (semantic):");   show_hits(dense_hits[:5])
print("\nBM25 (keyword):");   show_hits(bm25_hits[:5], "bm25")
print("\nHybrid (RRF fused):"); show_hits(hybrid_hits, "rrf")

Dense (semantic):
  1. Moby-Dick [moby-dick#123]  distance=0.484
  2. Moby-Dick [moby-dick#568]  distance=0.579
  3. Moby-Dick [moby-dick#648]  distance=0.587
  4. Moby-Dick [moby-dick#378]  distance=0.587
  5. Moby-Dick [moby-dick#815]  distance=0.590

BM25 (keyword):
  1. Moby-Dick [moby-dick#628]  bm25=13.507
  2. Moby-Dick [moby-dick#629]  bm25=13.060
  3. War and Peace [war-and-peace#2176]  bm25=13.004
  4. War and Peace [war-and-peace#2340]  bm25=12.811
  5. War and Peace [war-and-peace#655]  bm25=12.417

Hybrid (RRF fused):
  1. Moby-Dick [moby-dick#123]  rrf=0.016
  2. Moby-Dick [moby-dick#628]  rrf=0.016
  3. Moby-Dick [moby-dick#568]  rrf=0.016
  4. Moby-Dick [moby-dick#629]  rrf=0.016
  5. Moby-Dick [moby-dick#648]  rrf=0.016


## Conversational RAG 

In [23]:
#"Who is Mr. Darcy?"
#"Who does he eventually marry?"

In [24]:
history = []          # list of (role, text) turns
followups = ["Who is Mr. Darcy?",
             "Who does he eventually marry?",
             "Why did she dislike him at first?"]

for question in followups:
    # 1. Condense the follow-up into a standalone query (resolve pronouns) using the history.
    if history:
        convo = "\n".join(f"{role}: {txt}" for role, txt in history)
        prompt = ("Rewrite the follow-up as a standalone question (resolve pronouns/references). "
                  "Return ONLY the rewritten question.\n\n"
                  f"Conversation:\n{convo}\n\nFollow-up: {question}")
        standalone = ollama_chat([{"role": "user", "content": prompt}], temperature=0.0).strip()
    else:
        standalone = question

    # 2. Retrieve on the standalone query, 3. answer, 4. remember the turn.
    answer = generate(standalone, dense_search(standalone))
    history += [("user", question), ("assistant", answer)]

    print(f"\nYou: {question}")
    print(f"   (retrieved as -> {standalone})")
    print(textwrap.fill(f"Bot: {answer}", width=88))


You: Who is Mr. Darcy?
   (retrieved as -> Who is Mr. Darcy?)
Bot: Based on the provided context, Mr. Darcy is a man who possesses a large estate in
Derbyshire [pride-and-prejudice#30]. He is described as proud, above his company, and
having a forbidding countenance [pride-and-prejudice#30], leading some to declare him
the proudest, most disagreeable man in the world [pride-and-prejudice#30]. He has
splendid property, noble kindred, and extensive patronage [pride-and-prejudice#495], and
his aunt is Lady Catherine de Bourgh [pride-and-prejudice#495]. At a ball, he danced
only once with Mrs. Hurst and Miss Bingley [pride-and-prejudice#30] and told Mr. Bingley
that Elizabeth Bennet was "tolerable: but not handsome enough to tempt me" [pride-and-
prejudice#31]. Later, he found her conversation interesting at Sir William Lucas's
[pride-and-prejudice#47]. Additionally, he accused Mr. Wickham of extravagance and
general profligacy [pride-and-prejudice#288].

You: Who does he eventually marry

### Re-ranking  

In [14]:
# Try a real cross-encoder; fall back to the LLM reranker if the dep is missing.
try:
    from sentence_transformers import CrossEncoder
    cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    HAVE_CROSS_ENCODER = True
    print("Cross-encoder loaded.")
except Exception as e:
    cross_encoder, HAVE_CROSS_ENCODER = None, False
    print(f"Cross-encoder unavailable ({type(e).__name__}) — will use the LLM reranker.")

Cross-encoder unavailable (ModuleNotFoundError) — will use the LLM reranker.


In [21]:
query = "the detective examines a battered hat for clues about its owner"
pool = dense_search(query, k=12)          # retrieve wide, then re-rank down

print("Before re-ranking (dense top-5):")
show_hits(pool[:5])

if HAVE_CROSS_ENCODER:
    # Cross-encoder scores each (query, chunk) pair jointly.
    ce_scores = cross_encoder.predict([(query, h["text"]) for h in pool])
    order = np.argsort(ce_scores)[::-1][:TOPK]
    reranked = [{**pool[i], "rerank": float(ce_scores[i])} for i in order]
else:
    # LLM reranker: one tiny 0-10 rating call per candidate (keep the pool small).
    reranked = []
    for h in pool[:8]:
        out = ollama_chat([{"role": "user", "content":
            f"Rate 0-10 how well this passage helps answer the question. Reply with ONLY a number.\n"
            f"Question: {query}\nPassage: {h['text'][:1200]}"}], temperature=0.0)
        m = re.search(r"\d+(?:\.\d+)?", out)
        reranked.append({**h, "rerank": float(m.group()) if m else 0.0})
    reranked = sorted(reranked, key=lambda h: -h["rerank"])[:TOPK]

print("\nAfter re-ranking (top-5):")
show_hits(reranked, "rerank")

Before re-ranking (dense top-5):
  1. Adventures of Sherlock Holmes [adventures-of-sherlock-holmes#216]  distance=0.416
  2. Adventures of Sherlock Holmes [adventures-of-sherlock-holmes#219]  distance=0.500
  3. Adventures of Sherlock Holmes [adventures-of-sherlock-holmes#217]  distance=0.513
  4. Adventures of Sherlock Holmes [adventures-of-sherlock-holmes#89]  distance=0.530
  5. Adventures of Sherlock Holmes [adventures-of-sherlock-holmes#218]  distance=0.533


KeyboardInterrupt: 